# Lesson 13: principal component analysis on the Treasury curve

1. Which direction through the days has the most variance?
2. How many components does the yield curve need?
3. What does a component say about the curve?
4. When does PCA help clustering, and when does it fail?

Daily US Treasury yields for 11 maturities, 1 month to 30 years, 5,165 days from 9 February 2006 to 30 September 2026, in per cent. PCA runs on the unscaled levels, since all 11 columns share one unit; k-means then groups the days on the first two component scores.

Treasury yields: Federal Reserve Board H.15 via FRED, Federal Reserve Bank of St. Louis; snapshot 30 September 2026.

In [ ]:
import pathlib

import pandas as pd

DATA_URL = "https://digital-ai-finance.github.io/teaching-msc-data-science/data/"
SNAPSHOT = "2026-09-30"


def load(name):
    """Read a snapshot: "treasury-curve-daily"."""
    local = pathlib.Path("../data/market") / f"{SNAPSHOT}-{name}.csv"
    if local.exists():  # run from material/notebooks/ before the files are published
        return pd.read_csv(local)
    url = f"{DATA_URL}{SNAPSHOT}-{name}.csv"
    try:
        return pd.read_csv(url)
    except Exception as err:
        raise RuntimeError(f"{url} not reachable ({err}). Download the file in a browser, upload "
                           "it in the Colab file panel and read it with pd.read_csv.") from err


import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import KMeans
from sklearn.datasets import make_swiss_roll
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

SEED = 42
YEARS = [1 / 12, 0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30]


def normalise(components, columns):
    """Fix each component's sign: PC1 sums positive, PC2 30-year positive, PC3 1-month positive."""
    out = components.copy()
    for k in range(len(out)):
        if k == 0:
            ref = out[k].sum()
        elif k == 1:
            ref = out[k, columns.index("DGS30")]
        elif k == 2:
            ref = out[k, columns.index("DGS1MO")]
        else:
            ref = out[k, np.abs(out[k]).argmax()]
        out[k] *= np.sign(ref)
    return out


def day_text(ts):
    return f"{ts.day} {ts:%B %Y}"

## 1. Five days by hand

The 2-year and the 10-year yield on five days A to E, each minus its mean, in tenths of a percentage point; the values are stylised. The variance of the days along a unit direction $u$ is $u^\top \Sigma u$, and the direction with the largest variance is the eigenvector of $\Sigma$ with the largest eigenvalue.

In [ ]:
days = np.array([[-3, -2], [-1, 0], [0, 0], [1, 2], [3, 0]], dtype=float)  # A to E, centred
S = np.cov(days.T)
eigvals, eigvecs = np.linalg.eigh(S)
first = eigvecs[:, -1] * np.sign(eigvecs[0, -1])
print("covariance matrix:", S.tolist())
print(f"eigenvalues {eigvals[1]:.0f} and {eigvals[0]:.0f}; first direction ({first[0]:.3f}, {first[1]:.3f}), "
      f"{np.degrees(np.arctan2(first[1], first[0])):.1f} degrees from the 2-year axis")
print("scores on the first direction:", ", ".join(f"{z:.2f}" for z in days @ first))
angles = np.radians(np.arange(0, 180, 0.5))
units = np.column_stack([np.cos(angles), np.sin(angles)])
variance = np.einsum("ij,jk,ik->i", units, S, units)
for deg in (0, 26.565, 45, 90):
    u = np.array([np.cos(np.radians(deg)), np.sin(np.radians(deg))])
    print(f"variance along {deg:g} degrees: {u @ S @ u:.2f}")
plt.plot(np.degrees(angles), variance)
plt.axvline(np.degrees(np.arctan2(1, 2)), ls="--")
plt.xlabel("direction, degrees from the 2-year axis")
plt.ylabel("variance of the five days along it")
plt.show()

In [ ]:
coeffs = np.poly(S)  # characteristic polynomial of the covariance matrix
print("lambda^2 coefficients:", coeffs.round(6).tolist(), "roots:", np.roots(coeffs).round(6).tolist())
print(f"trace {np.trace(S):.0f} = sum of eigenvalues, determinant {np.linalg.det(S):.0f} = product")

**Exercise 1.** Set the covariance of the five days to 0 and keep the variances 5 and 2. Predict the curve of variance against angle before you plot it: where does it peak, and at what value?

## 2. The curve as a table

One row per day, one column per maturity. The variances run from about 4 at the short end to about 1 at the 30-year, all in per cent squared.

In [ ]:
curve = load("treasury-curve-daily")
curve["date"] = pd.to_datetime(curve["date"])
curve = curve.set_index("date")
cols = list(curve.columns)
X = curve.to_numpy()
print(curve.shape)
print(f"{len(curve):,} days, {day_text(curve.index[0])} to {day_text(curve.index[-1])}")
print("variances, per cent squared:", ", ".join(f"{c} {v:.2f}" for c, v in curve.var().items()))
for day in ["2006-02-09", "2023-07-03", "2026-09-30"]:
    plt.plot(YEARS, curve.loc[day], marker="o", label=day)
plt.xscale("log")
plt.xlabel("maturity, years")
plt.ylabel("yield, per cent")
plt.legend()
plt.show()

In [ ]:
pca = PCA(svd_solver="full").fit(X)
comps = normalise(pca.components_, cols)
lam = pca.explained_variance_
share = pca.explained_variance_ratio_
cum = np.cumsum(share)
ks = np.arange(1, len(share) + 1)
fig, ax = plt.subplots()
ax.bar(ks, 100 * share)
ax.set_yscale("log")
ax.set_xlabel("component")
ax.set_ylabel("share of variance, per cent (log scale)")
right = ax.twinx()
right.plot(ks, 100 * cum, color="tab:orange", marker="o")
right.set_ylabel("cumulative share, per cent")
plt.show()
print("eigenvalues:", ", ".join(f"{v:.3f}" for v in lam[:3]), f"of a total {lam.sum():.3f}")
print("shares, per cent:", ", ".join(f"{100 * v:.1f}" for v in share[:4]))
print("cumulative, per cent:", ", ".join(f"{100 * v:.1f}" for v in cum[:4]))
for level in (0.90, 0.95, 0.99):
    print(f"components for {level:.0%}:", int(np.searchsorted(cum, level) + 1))
standardised = PCA(svd_solver="full").fit(StandardScaler().fit_transform(X))
print("standardised columns, shares, per cent:",
      ", ".join(f"{100 * v:.1f}" for v in standardised.explained_variance_ratio_[:3]))

**Exercise 2.** Multiply the column `DGS30` by 100, the 30-year yield in basis points, and fit the PCA again. What share does the first component explain, and what weight does it put on the 30-year?

## 3. Loadings, biplot and scores

A component's loadings are its weights across the 11 maturities. A score is a day's position along a component.

In [ ]:
labels = ["1m", "3m", "6m", "1y", "2y", "3y", "5y", "7y", "10y", "20y", "30y"]
for k, name in enumerate(["PC1 level", "PC2 slope", "PC3 curvature"]):
    plt.plot(YEARS, comps[k], marker="o", label=name)
    print(f"{name}: " + ", ".join(f"{m} {w:.2f}" for m, w in zip(labels, comps[k])))
print(f"sum of the weights: PC1 {comps[0].sum():.2f}, PC2 {comps[1].sum():.2f}")
plt.axhline(0, color="grey", lw=0.8)
plt.xscale("log")
plt.xlabel("maturity, years")
plt.ylabel("weight")
plt.legend()
plt.show()

In [ ]:
mu = X.mean(axis=0)
scores = (X - mu) @ comps.T
arrows = comps[:2].T * np.sqrt(lam[:2]) / X.std(axis=0, ddof=1)[:, None]
angle = np.degrees(np.arctan2(arrows[:, 1], arrows[:, 0]))
gap = angle[cols.index("DGS30")] - angle[cols.index("DGS1MO")]
print(f"angle between the 1-month and the 30-year arrow: {gap:.1f} degrees, cosine {np.cos(np.radians(gap)):.2f}")
print(f"correlation 1-month and 3-month: {curve['DGS1MO'].corr(curve['DGS3MO']):.3f}; "
      f"1-month and 30-year: {curve['DGS1MO'].corr(curve['DGS30']):.3f}")
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(scores[:, 0], scores[:, 1], s=3, color="grey", alpha=0.3)
for name, (a, b) in zip(labels, arrows):
    ax.arrow(0, 0, 8 * a, 8 * b, head_width=0.15, color="black")
    ax.text(8.6 * a, 8.6 * b, name, fontsize=8)
ax.set_aspect("equal")
ax.set_xlabel("PC1 score")
ax.set_ylabel("PC2 score")
plt.show()

In [ ]:
mean_yield = X.mean(axis=1)
spread = curve["DGS10"] - curve["DGS3MO"]
print(f"PC1 score and mean yield: {np.corrcoef(scores[:, 0], mean_yield)[0, 1]:.3f}")
print(f"PC2 score and 10y - 3m spread: {np.corrcoef(scores[:, 1], spread)[0, 1]:.3f}")
fig, (top, bottom) = plt.subplots(2, 1, sharex=True, figsize=(8, 5))
top.plot(curve.index, scores[:, 0], label="PC1 score")
top.plot(curve.index, mean_yield, label="mean yield")
bottom.plot(curve.index, scores[:, 1], label="PC2 score")
bottom.plot(curve.index, spread, label="10y - 3m spread")
top.legend()
bottom.legend()
plt.show()

A component's sign is arbitrary: flipping every weight of a component, and every score on it, describes the same direction, and two library versions may return opposite signs. `normalise` fixes the signs here, PC1 summing positive, PC2 positive at 30 years, PC3 positive at 1 month. Compare signs inside one component only.

In [ ]:
day = curve.index.get_loc(pd.Timestamp("2026-09-30"))
plt.plot(YEARS, X[day], "ko", label="measured")
for k in (1, 2, 3):
    rebuilt = mu + scores[day, :k] @ comps[:k]
    miss = 100 * np.abs(rebuilt - X[day]).max()
    rmse = 100 * np.sqrt(((mu + scores[:, :k] @ comps[:k] - X) ** 2).mean())
    plt.plot(YEARS, rebuilt, label=f"{k} component{'s' if k > 1 else ''}")
    print(f"{k} component{'s' if k > 1 else ''}: largest miss {miss:.0f} bp on this day, RMSE {rmse:.1f} bp on all days")
plt.xscale("log")
plt.legend()
plt.show()

Xc = X - mu


def redraw(pair):
    """Share of the 11 yields' variance that least squares redraws from two columns."""
    A = curve[pair].to_numpy() - curve[pair].to_numpy().mean(axis=0)
    B = np.linalg.lstsq(A, Xc, rcond=None)[0]
    return 1 - ((Xc - A @ B) ** 2).sum() / (Xc ** 2).sum()


for name, pair in [("A, 2-year and 10-year", ["DGS2", "DGS10"]), ("B, 1-month and 3-month", ["DGS1MO", "DGS3MO"]),
                   ("6-month and 10-year", ["DGS6MO", "DGS10"])]:
    print(f"{name}: {100 * redraw(pair):.1f} per cent redrawn")
print(f"C, the first two components: {100 * cum[1]:.1f} per cent redrawn")

## 4. Regimes: PCA before k-means

k-means groups the days on their first two component scores, with `n_init=10` and `random_state=42`; the silhouette compares k = 2 to 6. The same k-means on all 11 yields shows how much the 9 dropped components change.

In [ ]:
S2 = scores[:, :2]
rows = []
for k in range(2, 7):
    a = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(S2).labels_
    b = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X).labels_
    table = pd.crosstab(a, b).to_numpy()
    r, c = linear_sum_assignment(-table)
    rows.append((k, silhouette_score(S2, a), silhouette_score(X, b), table[r, c].sum()))
sil = pd.DataFrame(rows, columns=["k", "two scores", "all 11 yields", "same regime"]).set_index("k")
print(sil.round(3))
for k, row in sil.iterrows():
    print(f"k = {k}: silhouette {row['two scores']:.3f} on two scores, {row['all 11 yields']:.3f} on all 11 yields; "
          f"{int(row['same regime']):,} days in the same regime")

In [ ]:
def regimes(k):
    model = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(S2)
    order = pd.Series(X[:, cols.index("DGS3MO")]).groupby(model.labels_).mean().sort_values().index
    rank = {label: i for i, label in enumerate(order)}
    return model, pd.Series([rank[l] for l in model.labels_], index=curve.index), rank


fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 2.6))
for ax, k in zip(axes, (2, 3)):
    _, lab, _ = regimes(k)
    ax.scatter(curve.index, np.zeros(len(lab)), c=lab, cmap="coolwarm", marker="|", s=200)
    ax.set_yticks([])
    ax.set_ylabel(f"k = {k}")
plt.show()
for k in (2, 3):
    _, lab, _ = regimes(k)
    profile = curve.groupby(lab.values)[["DGS3MO", "DGS2", "DGS10", "DGS30"]].mean().round(2)
    profile["days"] = lab.value_counts().sort_index().values
    print(f"k = {k}\n", profile)
    print("days per regime, low to high:", ", ".join(f"{n:,}" for n in profile["days"]))
inverted = (curve["DGS10"] < curve["DGS3MO"]).to_numpy()
runs = np.cumsum(np.r_[True, inverted[1:] != inverted[:-1]])
inv = runs == pd.Series(runs[inverted]).value_counts().idxmax()
print(f"longest inversion: {inv.sum()} days, {day_text(curve.index[inv][0])} to {day_text(curve.index[inv][-1])}; "
      f"regimes k = 2: {sorted(set(regimes(2)[1][inv]))}, k = 3: {sorted(set(regimes(3)[1][inv]))}")

In [ ]:
model, lab, rank = regimes(2)
new = (X[-1] - mu) @ comps[:2].T
print("30 September 2026, scores", new.round(2), "regime", rank[model.predict(new.reshape(1, -1))[0]],
      "(0 low, 1 high)")

**Exercise 3.** Divide the two scores by their standard deviations before k-means and compare the silhouettes for k = 2 to 6 with those on the unscaled scores. Which k wins now, and why does the scaling change the distances?

## 5. The zero floor

The 1-month yield stays at or above zero; a projection on two components has no such floor.

In [ ]:
rebuilt = mu + scores[:, :2] @ comps[:2]
rebuilt3 = mu + scores[:, :3] @ comps[:3]
one_month = X[:, cols.index("DGS1MO")]
below = rebuilt[:, 0] < 0
print(f"days at or below 0.05 per cent: {int((one_month <= 0.05).sum()):,}")
print(f"days reconstructed below zero: {int(below.sum()):,}, lowest {rebuilt[:, 0].min():.2f} "
      f"on {day_text(curve.index[rebuilt[:, 0].argmin()])} (measured {one_month[rebuilt[:, 0].argmin()]:.2f})")
print(f"three components: {int((rebuilt3[:, 0] < 0).sum()):,} days below zero, lowest {rebuilt3[:, 0].min():.2f}")
plt.scatter(one_month, rebuilt[:, 0], s=3, alpha=0.4)
plt.axhline(0, color="red")
plt.xlabel("1-month yield, measured")
plt.ylabel("1-month yield, two components")
plt.show()

## 6. Extensions: daily changes and the swiss roll

PCA of the daily changes splits the variance more evenly than PCA of the levels. The swiss roll is a rolled surface in three dimensions: its best plane puts points from different turns of the roll side by side.

In [ ]:
changes = curve.diff().dropna()
moves = PCA(svd_solver="full").fit(changes.to_numpy())
print(f"{len(changes):,} changes; shares, per cent:", ", ".join(f"{100 * v:.1f}" for v in moves.explained_variance_ratio_[:3]))

In [ ]:
roll, t = make_swiss_roll(n_samples=1000, noise=0.5, random_state=SEED)
Zs = StandardScaler().fit_transform(roll)
flat = PCA(n_components=2, svd_solver="full").fit(Zs)
plane = flat.transform(Zs)
_, idx = NearestNeighbors(n_neighbors=11).fit(plane).kneighbors(plane)
other = (np.abs(t[idx[:, 1:]] - t[:, None]) > np.pi).mean()
print(f"two components explain {flat.explained_variance_ratio_.sum():.1%}; "
      f"{other:.1%} of plane neighbours lie on another turn of the roll")
plt.scatter(plane[:, 0], plane[:, 1], c=t, s=6)
plt.colorbar(label="position along the roll, t")
plt.show()

## Newer data (optional)

FRED serves the 11 series live, up to its latest published day. The slides use the snapshot of 30 September 2026, 5,165 days; the live series count more days, and the shares may move from the slides' 92.2, 6.7 and 0.9 per cent as days are added.

In [ ]:
FRED = "https://fred.stlouisfed.org/graph/fredgraph.csv?id="
BIS = "https://stats.bis.org/api/v1/data/{}?startPeriod={}&format=csv"


def fred(*ids):
    """Live FRED series, e.g. fred("GS10", "TB3MS", "USREC"); "." marks a missing value."""
    return pd.read_csv(FRED + ",".join(ids), na_values=["."], parse_dates=["observation_date"])


def bis(flow, start):
    """Live BIS series, e.g. bis("WS_CBPOL/M..", "2021-12")."""
    return pd.read_csv(BIS.format(flow, start))


live = fred(*cols).dropna().set_index("observation_date").loc["2006-02-09":]  # the 30-year resumes in 2006
live_pca = PCA(svd_solver="full").fit(live.to_numpy())
print(f"{len(live):,} days to {live.index[-1].date()}; shares, per cent:",
      ", ".join(f"{100 * v:.1f}" for v in live_pca.explained_variance_ratio_[:3]))

## Two or three directions of largest variance redraw the Treasury curve

1. The direction with the most variance is the first principal component, the eigenvector of the covariance matrix with the largest eigenvalue: $(2, 1)/\sqrt5$ with variance 6 for the five days.
2. Two components explain 98.9 per cent of the curve's variance and three 99.8 per cent, and three redraw the 11 yields within 7.1 basis points on average.
3. PC1 lifts every yield (level), PC2 tilts the short end against the long end (slope) and PC3 bends the middle against both ends (curvature); columns in different units are standardised before PCA.
4. k-means on the first two component scores puts all 5,165 days in the regime k-means finds on all 11 yields at k = 2. At the zero floor the measured 1-month yield sat at or below 0.05 per cent on 1,277 days, never below zero; two components push it below zero on 1,376 of all 5,165 days.